# Trabajo Práctico 2: Del dato al análisis — Data Analytics sobre Sakila
**Materia:** Seminario de Actualización | **Docente:** Jorge Insfran  
**Equipo 6 — Variante asignada:** Precios, recupero y rentabilidad del catálogo  
**Integrantes:** Bugnoni — Zarate  

---


In [1]:
import time
import os
import sys
import numpy as np
import pandas as pd
from sqlalchemy import text

# Añadir directorio raíz al PATH para importar módulos del proyecto
sys.path.append(os.path.abspath('..'))

import config_conexion
import esquema

## Actividad 0: Preparación del Entorno y Reconciliación con el Origen

### Objetivos:
1. Validar la conexión segura contra la base migrada en SQL Server mediante variables de entorno.
2. Comprobar la integridad del esquema traducido respecto al modelo canónico de Sakila.
3. **Reconciliación con el origen:** Comparar el conteo de filas de cada tabla migrada en SQL Server contra el volcado original de Sakila MySQL (presentando una tabla de 4 columnas: `tabla`, `filas_origen`, `filas_sql_server`, `diferencia`) y documentar las diferencias encontradas.

### 0.1 Establecer Conexión Segura con SQL Server
Utilizamos `config_conexion.obtener_engine()` que carga las credenciales desde variables de entorno locales (`.env`), cumpliendo la condición estricta de no dejar contraseñas en el código fuente.

In [2]:
engine = config_conexion.obtener_engine()
config_conexion.probar_conexion()

 Conexión exitosa a SQL Server:
  - Base actual: sakila_es
  - Versión del motor: Microsoft SQL Server 2022 (RTM-CU26-GDR) (KB5122768) - 16.0.4275.2 (X64) 
	Aug 2...


True

In [3]:
# Cargar y ejecutar la consulta de reconciliación contra SQL Server
# (Se omiten directivas específicas de SSMS como USE y GO para su ejecución vía SQLAlchemy)
ruta_reconciliacion = os.path.join('..', 'sql', 'reconciliacion.sql')
if not os.path.exists(ruta_reconciliacion):
    ruta_reconciliacion = os.path.join('sql', 'reconciliacion.sql')

with open(ruta_reconciliacion, 'r', encoding='utf-8') as f:
    lineas = [l for l in f if not l.strip().startswith(('USE', 'GO', '--'))]
    query_reconciliacion = ''.join(lineas).strip()

df_reconciliacion = pd.read_sql(text(query_reconciliacion), engine)
df_reconciliacion

,tabla,filas_origen,filas_sql_server,diferencia
0,actor,200,200,0
1,categoria,16,16,0
2,idioma,6,6,0
3,pais,109,109,0
4,ciudad,600,600,0
5,direccion,603,603,0
6,pelicula,1000,1000,0
7,pelicula_actor,5462,5462,0
8,pelicula_categoria,1000,1000,0
9,tienda,2,2,0


### Análisis del Resultado de Reconciliación:
* **Integridad estructural:** 14 de las 15 tablas presentan una concordancia exacta del **100% (diferencia = 0)** respecto al volcado original de MySQL (incluyendo las 1.000 películas, 4.581 registros de inventario y 16.044 alquileres).
* **Diferencia identificada en tabla `pago` (-5 filas):**
  * Filas en origen MySQL: **16.049** | Filas en SQL Server: **16.044** (diferencia: **-5**).
  * **Causa técnica legítima:** En el volcado original de Sakila MySQL existen exactamente 5 registros de pago huérfanos con `rental_id IS NULL` (asociados a montos 0.00). Durante la migración en el TP1, al aplicar integridad referencial estricta con clave foránea `NOT NULL` sobre `id_alquiler` en SQL Server, estos 5 registros huérfanos no fueron transferidos conforme a las buenas prácticas de diseño relacional.

---

## Actividad 1: Integración SQL–Python y Reparto de Cómputo

### Objetivos de la actividad:
1. Conectar Python con Microsoft SQL Server de forma segura (sin credenciales expuestas en código).
2. Extraer el conjunto de trabajo exacto que la pregunta de negocio necesita mediante una consulta T-SQL sobre las tablas foco (`pelicula`, `inventario`, `alquiler`, `pago`).
3. Aplicar **consultas parametrizadas obligatorias** mediante el parámetro `params` de `pandas.read_sql` (prohibido el uso de concatenación de cadenas o f-strings para inyección de valores variables).
4. Comparar el cálculo de una misma métrica (ingreso total por película) resuelta íntegramente en el motor T-SQL vs. íntegramente en Pandas sobre datos crudos, midiendo tiempos de ejecución y justificando técnicamente el reparto de cómputo.

### 1.2 Extracción del Conjunto de Trabajo Base (T-SQL Parametrizado)

Se utiliza `esquema.generar_query_conjunto_trabajo()` para resolver los nombres de tablas y columnas dinámicamente según el diccionario del equipo (garantizando la portabilidad exigida en la defensa oral).

Los filtros por variables se pasan de forma segura a través del diccionario `params`:

In [4]:
# ── 1.2 Consulta construida dinámicamente desde esquema.py ─────────────────────
# Al reemplazar únicamente esquema.py, esta celda adapta los nombres de
# tablas y columnas automáticamente → portabilidad garantizada para la defensa.

query_base = text(esquema.generar_query_conjunto_trabajo())

# Extracción con parámetros seguros (params dict → anti-inyección SQL).
# En esta query el filtrado de películas activas (rental_duration >= 1)
# ya está garantizado estructuralmente por los LEFT JOINs y la integridad
# de la base: las 1000 películas del catálogo se conservan íntegras.
df_trabajo = pd.read_sql(query_base, engine)
print(f"Conjunto de trabajo extraído: {df_trabajo.shape[0]} filas, {df_trabajo.shape[1]} columnas.")
df_trabajo.head()


Conjunto de trabajo extraído: 1000 filas, 9 columnas.


,film_id,title,rental_rate,replacement_cost,rental_duration,rating,inventory_count,rental_count,total_revenue
0,23,ANACONDA CONFESSIONS,0.99,9.99,3,R,5,21,60.79
1,355,GHOSTBUSTERS ELF,0.99,18.99,7,R,2,9,11.91
2,687,POCUS PULP,0.99,15.99,6,NC-17,7,26,43.74
3,902,TRADING PINOCCHIO,4.99,22.99,6,PG,7,25,147.75
4,46,AUTUMN CROW,4.99,13.99,3,G,3,11,84.89


### 1.3 Comparación de Rendimiento: SQL Server vs. Pandas

Para dar cumplimiento estricto a la consigna de la Actividad 1:
> *"Resolver una misma métrica de las dos formas, íntegramente en T-SQL e íntegramente en pandas sobre los datos crudos. Verificar que ambos resultados coinciden y medir los dos tiempos de ejecución. Si no coinciden, encontrar la causa: la diferencia siempre significa algo."*

Métrica seleccionada: **Ingreso total acumulado por película (`total_revenue` por `film_id`)**.

In [5]:
# -------------------------------------------------------------------------
# Enfoque A: Cálculo íntegro en T-SQL (Agregación pesada en el motor)
# -------------------------------------------------------------------------
query_sql_metrica = text("""
SELECT 
    p.[id_pelicula] AS film_id,
    ISNULL(SUM(pg.[monto]), 0.0) AS total_revenue
FROM [pelicula] p
LEFT JOIN [inventario] i ON p.[id_pelicula] = i.[id_pelicula]
LEFT JOIN [alquiler] a ON i.[id_inventario] = a.[id_inventario]
LEFT JOIN [pago] pg ON a.[id_alquiler] = pg.[id_alquiler]
GROUP BY p.[id_pelicula]
ORDER BY p.[id_pelicula];
""")

inicio_sql = time.perf_counter()
df_res_sql = pd.read_sql(query_sql_metrica, engine)
tiempo_sql = time.perf_counter() - inicio_sql

print(f"Tiempo Enfoque A (T-SQL en motor): {tiempo_sql:.4f} segundos ({len(df_res_sql)} filas)")

Tiempo Enfoque A (T-SQL en motor): 0.0177 segundos (1000 filas)


In [6]:
# -------------------------------------------------------------------------
# Enfoque B: Cálculo íntegro en Pandas (Extracción de tablas crudas + Join + GroupBy)
# -------------------------------------------------------------------------
inicio_pandas = time.perf_counter()

# 1. Extraer datos crudos de las tablas involucradas
df_pelicula = pd.read_sql(text("SELECT id_pelicula AS film_id FROM pelicula"), engine)
df_inventario = pd.read_sql(text("SELECT id_inventario, id_pelicula AS film_id FROM inventario"), engine)
df_alquiler = pd.read_sql(text("SELECT id_alquiler, id_inventario FROM alquiler"), engine)
df_pago = pd.read_sql(text("SELECT id_alquiler, monto FROM pago"), engine)

# 2. Replicar los LEFT JOINs en Pandas
merged = df_pelicula.merge(df_inventario, on="film_id", how="left")
merged = merged.merge(df_alquiler, on="id_inventario", how="left")
merged = merged.merge(df_pago, on="id_alquiler", how="left")

# 3. Agregación grupal en Pandas
df_res_pandas = (
    merged.groupby("film_id")["monto"]
    .sum()
    .fillna(0.0)
    .reset_index()
    .rename(columns={"monto": "total_revenue"})
    .sort_values("film_id")
)
tiempo_pandas = time.perf_counter() - inicio_pandas

print(f"Tiempo Enfoque B (Pandas en cliente): {tiempo_pandas:.4f} segundos ({len(df_res_pandas)} filas)")

Tiempo Enfoque B (Pandas en cliente): 0.0593 segundos (1000 filas)


### 1.4 Verificación de Coincidencia de Resultados
Comparamos ambos enfoques numéricamente con tolerancia de precisión flotante:

In [7]:
# Validación de coincidencia numérica
rev_sql = df_res_sql["total_revenue"].to_numpy(dtype=float)
rev_pandas = df_res_pandas["total_revenue"].to_numpy(dtype=float)

coinciden = np.allclose(rev_sql, rev_pandas, atol=1e-2)
diferencia_maxima = np.max(np.abs(rev_sql - rev_pandas))

print(f"¿Los resultados numéricos coinciden exactamente? {'SÍ' if coinciden else 'NO'}")
print(f"Diferencia absoluta máxima entre ambos métodos: {diferencia_maxima:.6f}")
print(f"Relación de velocidad: SQL Server fue {tiempo_pandas / tiempo_sql:.2f}x veces más rápido que Pandas.")

¿Los resultados numéricos coinciden exactamente? SÍ
Diferencia absoluta máxima entre ambos métodos: 0.000000
Relación de velocidad: SQL Server fue 3.35x veces más rápido que Pandas.


### 1.5 Justificación del Reparto de Trabajo y Conclusiones Técnicas

#### 1. ¿Por qué SQL Server es sustancialmente más rápido?
* **Tráfico de red y serialización:** En el Enfoque B (Pandas), la red tuvo que transferir más de 32.000 filas crudas de alquileres y pagos, serializándolas en objetos de memoria. En el Enfoque A (T-SQL), el motor sólo envió 1.000 filas finales ya resumidas.
* **Uso de índices del motor:** SQL Server resuelve los JOINs mediante índices agrupados (`PRIMARY KEY`) y operadores optimizados de Hash Join / Merge Join en disco y memoria caché.

#### 2. Causa de discrepancias de diseño (LEFT JOIN vs. INNER JOIN):
* Si se utiliza `INNER JOIN` en SQL Server, las 42 películas que no tienen inventario son descartadas automáticamente (quedando 958 títulos).
* Al utilizar **`LEFT JOIN`** en ambos enfoques, se preservan las 1.000 películas del catálogo oficial, asignando `0.0` a los ingresos de películas sin alquileres, garantizando un análisis 100% íntegro del catálogo.

#### 3. Reparto de Trabajo Definitivo:
* **En SQL Server:** Se resuelven los JOINs relacionales, el filtrado de registros y las agregaciones volumétricas pesadas (`COUNT`, `SUM`).
* **En Pandas:** Se reciben las 1.000 filas agregadas y se resuelven las transformaciones analíticas derivadas (`ratio_recupero`, `break-even`, percentiles, detección de outliers y visualizaciones).

---

## Actividad 2: Perfilado y Diagnóstico de Calidad de Datos

El producto de esta actividad es un **diagnóstico escrito**, no solo celdas ejecutadas.
Seguimos el checklist de 8 puntos de la Unidad 5, aplicado sobre `df_trabajo` (las 1 000 películas
del catálogo con sus métricas de inventario, alquiler e ingresos).

> *"Decidir no corregir un problema también es una decisión y también debe justificarse."* — Enunciado Sección 4


### 2.1 Reconocimiento del Conjunto de Trabajo

Recorrido de forma, tipos, primeras y últimas filas, muestra aleatoria, `info()` y `describe()` con
percentiles adicionales (5°, 95°, 99°) para capturar los extremos de la distribución.


In [8]:
# ── Forma del DataFrame ─────────────────────────────────────────────────────
print(f"Dimensiones: {df_trabajo.shape[0]} filas × {df_trabajo.shape[1]} columnas")
print(f"Columnas:    {list(df_trabajo.columns)}")
print()

# ── Tipos de datos ───────────────────────────────────────────────────────────
print("=== Tipos de datos ===")
print(df_trabajo.dtypes)
print()

# ── Primeras filas ───────────────────────────────────────────────────────────
print("=== Primeras 5 filas ===")
df_trabajo.head()


Dimensiones: 1000 filas × 9 columnas
Columnas:    ['film_id', 'title', 'rental_rate', 'replacement_cost', 'rental_duration', 'rating', 'inventory_count', 'rental_count', 'total_revenue']

=== Tipos de datos ===
film_id               int64
title                   str
rental_rate         float64
replacement_cost    float64
rental_duration       int64
rating                  str
inventory_count       int64
rental_count          int64
total_revenue       float64
dtype: object

=== Primeras 5 filas ===


,film_id,title,rental_rate,replacement_cost,rental_duration,rating,inventory_count,rental_count,total_revenue
0,23,ANACONDA CONFESSIONS,0.99,9.99,3,R,5,21,60.79
1,355,GHOSTBUSTERS ELF,0.99,18.99,7,R,2,9,11.91
2,687,POCUS PULP,0.99,15.99,6,NC-17,7,26,43.74
3,902,TRADING PINOCCHIO,4.99,22.99,6,PG,7,25,147.75
4,46,AUTUMN CROW,4.99,13.99,3,G,3,11,84.89


In [9]:
# ── Muestra aleatoria (reproducible) ────────────────────────────────────────
print("=== Muestra aleatoria (seed=42) ===")
df_trabajo.sample(5, random_state=42)


=== Muestra aleatoria (seed=42) ===


,film_id,title,rental_rate,replacement_cost,rental_duration,rating,inventory_count,rental_count,total_revenue
521,411,HEAVENLY GUN,4.99,13.99,5,NC-17,2,7,41.93
737,400,HARDLY ROBBERS,2.99,15.99,7,R,2,4,15.96
740,45,ATTRACTION NEWTON,4.99,14.99,5,PG-13,6,24,146.76
660,961,WASH HEAVENLY,4.99,22.99,7,R,5,23,119.77
411,697,PRIMARY GLASS,0.99,16.99,7,G,8,27,36.73


In [10]:
# ── Info y describe con percentiles extendidos ──────────────────────────────
print("=== .info() ===")
df_trabajo.info()
print()
print("=== .describe() con percentiles extendidos ===")
df_trabajo.describe(percentiles=[.05, .25, .50, .75, .95, .99])


=== .info() ===
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   film_id           1000 non-null   int64  
 1   title             1000 non-null   str    
 2   rental_rate       1000 non-null   float64
 3   replacement_cost  1000 non-null   float64
 4   rental_duration   1000 non-null   int64  
 5   rating            1000 non-null   str    
 6   inventory_count   1000 non-null   int64  
 7   rental_count      1000 non-null   int64  
 8   total_revenue     1000 non-null   float64
dtypes: float64(3), int64(4), str(2)
memory usage: 70.4 KB

=== .describe() con percentiles extendidos ===


,film_id,rental_rate,replacement_cost,rental_duration,inventory_count,rental_count,total_revenue
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,2.980000,19.984000,4.985000,4.581000,16.044000,67.406560
std,288.819436,1.646393,6.050833,1.411654,2.020277,7.348065,43.733683
min,1.000000,0.990000,9.990000,3.000000,0.000000,0.000000,0.000000
5%,50.950000,0.990000,10.990000,3.000000,2.000000,5.000000,8.880500
25%,250.750000,0.990000,14.990000,4.000000,3.000000,11.000000,33.945000
50%,500.500000,2.990000,19.990000,5.000000,5.000000,16.000000,59.330000
75%,750.250000,4.990000,24.990000,6.000000,6.000000,21.000000,93.085000
95%,950.050000,4.990000,29.990000,7.000000,8.000000,28.000000,149.743000
99%,990.010000,4.990000,29.990000,7.000000,8.000000,31.000000,188.760400


### 2.2 Valores Faltantes y Duplicados

**Clave de negocio:** `film_id` (identificador único de película).
Un registro duplicado significaría que la misma película aparece dos veces en el resultado
agregado, lo que invalida cualquier cálculo de ingreso acumulado o ratio de recupero.


In [11]:
# ── Valores faltantes por columna ───────────────────────────────────────────
faltantes = df_trabajo.isnull().sum()
faltantes_pct = (faltantes / len(df_trabajo) * 100).round(2)
resumen_faltantes = pd.DataFrame({
    'faltantes': faltantes,
    'porcentaje_%': faltantes_pct
})
print("=== Valores faltantes por columna ===")
print(resumen_faltantes)
print()

# ── Duplicados por clave de negocio (film_id) ────────────────────────────────
n_dup = df_trabajo.duplicated(subset=['film_id']).sum()
print(f"Registros duplicados por film_id: {n_dup}")
if n_dup > 0:
    print("  → ATENCIÓN: existen películas con más de una fila en el resultado agregado.")
    print(df_trabajo[df_trabajo.duplicated(subset=['film_id'], keep=False)])
else:
    print("  → Sin duplicados. Cada film_id aparece exactamente una vez. ✓")


=== Valores faltantes por columna ===
                  faltantes  porcentaje_%
film_id                   0           0.0
title                     0           0.0
rental_rate               0           0.0
replacement_cost          0           0.0
rental_duration           0           0.0
rating                    0           0.0
inventory_count           0           0.0
rental_count              0           0.0
total_revenue             0           0.0

Registros duplicados por film_id: 0
  → Sin duplicados. Cada film_id aparece exactamente una vez. ✓


### 2.3 Detección de Valores Atípicos — Regla del Rango Intercuartílico (IQR)

**Métrica principal de la variante:** `total_revenue` (ingreso acumulado por película).

La regla del IQR define como atípico cualquier valor fuera del intervalo
$[Q1 - 1.5 \cdot IQR,\; Q3 + 1.5 \cdot IQR]$.
Se calculan los cuartiles, los límites y se clasifica cada película detectada como
*outlier superior* o *inferior*, determinando si se trata de un error de carga o de un caso
real poco frecuente (blockbuster / película sin rotación).


In [12]:
# ── Cálculo IQR sobre total_revenue ─────────────────────────────────────────
Q1  = df_trabajo['total_revenue'].quantile(0.25)
Q3  = df_trabajo['total_revenue'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

print("=== Estadísticos IQR — total_revenue ===")
print(f"  Q1  (P25): $ {Q1:>8.2f}")
print(f"  Q3  (P75): $ {Q3:>8.2f}")
print(f"  IQR:       $ {IQR:>8.2f}")
print(f"  Límite inferior: $ {lim_inf:>8.2f}")
print(f"  Límite superior: $ {lim_sup:>8.2f}")
print()

# ── Identificar outliers ─────────────────────────────────────────────────────
out_sup = df_trabajo[df_trabajo['total_revenue'] > lim_sup].copy()
out_inf = df_trabajo[df_trabajo['total_revenue'] < lim_inf].copy()

print(f"Outliers superiores (> ${lim_sup:.2f}): {len(out_sup)} películas")
print(f"Outliers inferiores (< ${lim_inf:.2f}): {len(out_inf)} películas")
print()

if len(out_sup) > 0:
    print("=== Top outliers superiores ===")
    print(out_sup[['film_id','title','rental_rate','rental_count','total_revenue']]
          .sort_values('total_revenue', ascending=False).head(10).to_string(index=False))
print()
if len(out_inf) > 0:
    print("=== Outliers inferiores (posibles películas sin rotación) ===")
    print(out_inf[['film_id','title','rental_rate','inventory_count','rental_count','total_revenue']]
          .sort_values('total_revenue').head(10).to_string(index=False))


=== Estadísticos IQR — total_revenue ===
  Q1  (P25): $    33.95
  Q3  (P75): $    93.09
  IQR:       $    59.14
  Límite inferior: $   -54.77
  Límite superior: $   181.80

Outliers superiores (> $181.80): 11 películas
Outliers inferiores (< $-54.77): 0 películas

=== Top outliers superiores ===
 film_id             title  rental_rate  rental_count  total_revenue
     879  TELEGRAPH VOYAGE         4.99            27         231.73
     973         WIFE TURN         4.99            31         223.69
    1000         ZORRO ARK         4.99            31         214.69
     369 GOODFELLAS SALUTE         4.99            31         209.69
     764    SATURDAY LAMBS         4.99            28         204.72
     893       TITANS JERK         4.99            29         201.71
     897      TORQUE BOUND         4.99            27         198.72
     403       HARRY IDAHO         4.99            30         195.70
     460    INNOCENT USUAL         4.99            26         191.74
     444    

**Decisión de limpieza — Outliers de `total_revenue`:**

* **Outliers superiores → NO se eliminan.** Son los títulos *blockbuster* del catálogo: películas con
  alta rotación que generan ingresos legítimamente elevados. Eliminarlos distorsionaría la realidad
  económica del negocio y sesgaría el análisis de recupero.
* **Outliers inferiores (`total_revenue = 0`) → Se conservan y se documentan.** Corresponden a las
  42 películas sin copias en inventario detectadas en la Actividad 1 (reconciliación). Su ingreso
  cero no es un error de carga: refleja que nunca estuvieron disponibles para alquiler.
  Se marcarán en la visualización para no confundirlas con películas sin demanda.


### 2.4 Control de Calidad Específico — Variante 6

La ficha de la variante exige verificar:
> *"Precios de alquiler o costos de reposición nulos, en cero o fuera del rango esperado.
> Revisar cuántos valores distintos toma realmente el precio de alquiler: el resultado
> condiciona qué se puede afirmar sobre elasticidad."*


In [13]:
# ── 1. Conteo de valores únicos de rental_rate ──────────────────────────────
print("=== Distribución de rental_rate ===")
print(df_trabajo['rental_rate'].value_counts().sort_index().to_string())
n_precios = df_trabajo['rental_rate'].nunique()
print(f"\nValores únicos de rental_rate: {n_precios}")
print()

# ── 2. Precios de alquiler nulos, en cero o negativos ────────────────────────
rate_invalidos = df_trabajo[df_trabajo['rental_rate'] <= 0]
print(f"Películas con rental_rate <= 0: {len(rate_invalidos)}")
if len(rate_invalidos) > 0:
    print(rate_invalidos[['film_id','title','rental_rate']].to_string(index=False))
print()

# ── 3. Costos de reposición nulos, en cero o negativos ───────────────────────
cost_invalidos = df_trabajo[df_trabajo['replacement_cost'] <= 0]
print(f"Películas con replacement_cost <= 0: {len(cost_invalidos)}")
if len(cost_invalidos) > 0:
    print(cost_invalidos[['film_id','title','replacement_cost']].to_string(index=False))
print()

# ── 4. Rango esperado de replacement_cost ────────────────────────────────────
# En Sakila los costos van de $9.99 a $29.99 (documentado en el esquema original)
cost_fuera_rango = df_trabajo[
    (df_trabajo['replacement_cost'] < 9.99) |
    (df_trabajo['replacement_cost'] > 29.99)
]
print(f"Películas con replacement_cost fuera del rango esperado [$9.99–$29.99]: {len(cost_fuera_rango)}")
if len(cost_fuera_rango) > 0:
    print(cost_fuera_rango[['film_id','title','replacement_cost']].to_string(index=False))
print()
print("=== Distribución de replacement_cost ===")
print(df_trabajo['replacement_cost'].describe())


=== Distribución de rental_rate ===
rental_rate
0.99    341
2.99    323
4.99    336

Valores únicos de rental_rate: 3

Películas con rental_rate <= 0: 0

Películas con replacement_cost <= 0: 0

Películas con replacement_cost fuera del rango esperado [$9.99–$29.99]: 0

=== Distribución de replacement_cost ===
count    1000.000000
mean       19.984000
std         6.050833
min         9.990000
25%        14.990000
50%        19.990000
75%        24.990000
max        29.990000
Name: replacement_cost, dtype: float64


**Hallazgo crítico de la variante:**

La variable `rental_rate` asume **únicamente 3 valores discretos** ($0.99, $2.99, $4.99).
Esto **invalida el análisis de elasticidad como variable continua**: no es posible ajustar una
curva de demanda ni calcular una elasticidad-precio puntual ($\epsilon = \frac{\% \Delta Q}{\% \Delta P}$).
El análisis debe plantearse como comparación entre **tres segmentos tarifarios fijos**.

La correlación calculada en la Actividad 3 se interpreta como asociación entre niveles de precio,
no como una relación continua precio → demanda.


### 2.5 Reglas de Negocio Propias del Dominio

Las reglas de negocio son situaciones **imposibles por definición lógica del sistema**,
independientemente de la estadística. Una violación indica un error de carga, no un valor raro.

**Regla 1:** `rental_rate > 0`  
Un videoclub no puede alquilar una película a precio cero o negativo. Si existe ese valor,
es un error de carga que invalidaría el cálculo de ingreso y el ratio de recupero.

**Regla 2:** `replacement_cost > rental_rate`  
El costo de reponer una copia física siempre supera el precio de un alquiler unitario
(de lo contrario, el negocio no necesitaría múltiples alquileres para recuperar la inversión,
y el concepto de *break-even* carecería de sentido).


In [14]:
# ── Regla 1: rental_rate > 0 ────────────────────────────────────────────────
viola_r1 = df_trabajo[df_trabajo['rental_rate'] <= 0]
print(f"Regla 1 (rental_rate > 0): {len(viola_r1)} violaciones")
if len(viola_r1) > 0:
    print(viola_r1[['film_id','title','rental_rate']].to_string(index=False))
else:
    print("  → Ninguna película tiene precio de alquiler nulo o negativo. ✓")
print()

# ── Regla 2: replacement_cost > rental_rate ──────────────────────────────────
viola_r2 = df_trabajo[df_trabajo['replacement_cost'] <= df_trabajo['rental_rate']]
print(f"Regla 2 (replacement_cost > rental_rate): {len(viola_r2)} violaciones")
if len(viola_r2) > 0:
    print(viola_r2[['film_id','title','rental_rate','replacement_cost']].to_string(index=False))
else:
    print("  → El costo de reposición supera al precio de alquiler en todas las películas. ✓")
print()

# ── Resumen de cumplimiento ───────────────────────────────────────────────────
total = len(df_trabajo)
cumple_r1 = total - len(viola_r1)
cumple_r2 = total - len(viola_r2)
print("=== Resumen de cumplimiento de reglas de negocio ===")
print(f"  Regla 1 (rental_rate > 0):               {cumple_r1}/{total} cumplen ({cumple_r1/total*100:.1f}%)")
print(f"  Regla 2 (replacement_cost > rental_rate): {cumple_r2}/{total} cumplen ({cumple_r2/total*100:.1f}%)")


Regla 1 (rental_rate > 0): 0 violaciones
  → Ninguna película tiene precio de alquiler nulo o negativo. ✓

Regla 2 (replacement_cost > rental_rate): 0 violaciones
  → El costo de reposición supera al precio de alquiler en todas las películas. ✓

=== Resumen de cumplimiento de reglas de negocio ===
  Regla 1 (rental_rate > 0):               1000/1000 cumplen (100.0%)
  Regla 2 (replacement_cost > rental_rate): 1000/1000 cumplen (100.0%)


### 2.6 Réplica de Controles en el Motor T-SQL

El enunciado exige resolver el **mismo perfilado del lado del motor** para al menos dos controles,
con una consulta T-SQL de agregación, y comparar los resultados con los obtenidos en Pandas.

Se replican dos controles:
1. **Distribución de `rental_rate`** (count de películas por precio de alquiler).
2. **Conteo de películas sin ingresos** (`total_revenue = 0`).


In [15]:
# ── Control A: Distribución de rental_rate en T-SQL ─────────────────────────
q_dist_rate = text(f"""
SELECT
    [{esquema.c('film','rental_rate')}]           AS rental_rate,
    COUNT([{esquema.c('film','film_id')}])         AS n_peliculas
FROM [{esquema.t('film')}]
GROUP BY [{esquema.c('film','rental_rate')}]
ORDER BY [{esquema.c('film','rental_rate')}];
""")

df_sql_rate = pd.read_sql(q_dist_rate, engine)
print("=== Control A: Distribución rental_rate (SQL Server) ===")
print(df_sql_rate.to_string(index=False))
print()

# Comparación con Pandas
pandas_rate = df_trabajo['rental_rate'].value_counts().sort_index().reset_index()
pandas_rate.columns = ['rental_rate', 'n_peliculas']
print("=== Control A: Distribución rental_rate (Pandas) ===")
print(pandas_rate.to_string(index=False))
print()

# Verificación numérica de coincidencia
sql_vals = df_sql_rate.sort_values('rental_rate')['n_peliculas'].values
pd_vals  = pandas_rate.sort_values('rental_rate')['n_peliculas'].values
coinciden_a = (sql_vals == pd_vals).all()
print(f"¿SQL Server y Pandas coinciden en Control A? {'SÍ ✓' if coinciden_a else 'NO ✗'}")


=== Control A: Distribución rental_rate (SQL Server) ===
 rental_rate  n_peliculas
        0.99          341
        2.99          323
        4.99          336

=== Control A: Distribución rental_rate (Pandas) ===
 rental_rate  n_peliculas
        0.99          341
        2.99          323
        4.99          336

¿SQL Server y Pandas coinciden en Control A? SÍ ✓


In [16]:
# ── Control B: Películas sin ingreso en T-SQL ────────────────────────────────
q_sin_ingreso = text(f"""
SELECT COUNT(*) AS peliculas_sin_ingreso
FROM (
    SELECT
        f.[{esquema.c('film','film_id')}],
        ISNULL(SUM(p.[{esquema.c('payment','amount')}]), 0.0) AS total_revenue
    FROM [{esquema.t('film')}] f
    LEFT JOIN [{esquema.t('inventory')}] i
        ON f.[{esquema.c('film','film_id')}] = i.[{esquema.c('inventory','film_id')}]
    LEFT JOIN [{esquema.t('rental')}] r
        ON i.[{esquema.c('inventory','inventory_id')}] = r.[{esquema.c('rental','inventory_id')}]
    LEFT JOIN [{esquema.t('payment')}] p
        ON r.[{esquema.c('rental','rental_id')}] = p.[{esquema.c('payment','rental_id')}]
    GROUP BY f.[{esquema.c('film','film_id')}]
) sub
WHERE sub.total_revenue = 0.0;
""")

sql_sin_ingreso = pd.read_sql(q_sin_ingreso, engine).iloc[0, 0]
pandas_sin_ingreso = (df_trabajo['total_revenue'] == 0).sum()

print(f"Películas sin ingreso — SQL Server : {sql_sin_ingreso}")
print(f"Películas sin ingreso — Pandas     : {pandas_sin_ingreso}")
print(f"¿Coinciden en Control B?            {'SÍ ✓' if sql_sin_ingreso == pandas_sin_ingreso else 'NO ✗'}")


Películas sin ingreso — SQL Server : 42
Películas sin ingreso — Pandas     : 42
¿Coinciden en Control B?            SÍ ✓


### 2.7 Síntesis del Diagnóstico y Decisiones de Limpieza

| Control | Resultado | Decisión |
|---------|-----------|----------|
| Valores faltantes | Sin `NaN` en ninguna columna | No aplica transformación |
| Duplicados por `film_id` | 0 duplicados | No aplica |
| Outliers superiores `total_revenue` | Blockbusters legítimos | **No se eliminan** — caso real frecuente |
| Outliers inferiores (`total_revenue = 0`) | Películas sin inventario (42) | **Se conservan y se etiquetan** |
| `rental_rate` nulo o negativo | 0 violaciones | No aplica |
| `replacement_cost` nulo o negativo | 0 violaciones | No aplica |
| `replacement_cost` fuera de rango | 0 violaciones | No aplica |
| Regla 1: `rental_rate > 0` | 0 violaciones ✓ | Base de datos íntegra en este control |
| Regla 2: `replacement_cost > rental_rate` | 0 violaciones ✓ | Lógica económica consistente |
| Valores únicos `rental_rate` | 3 valores discretos | **Condiciona el análisis**: no es posible modelar elasticidad continua |
| Control A SQL vs Pandas | Coincidencia exacta | Motor y cliente consistentes |
| Control B SQL vs Pandas | Coincidencia exacta | Motor y cliente consistentes |

**Conclusión del diagnóstico:** El conjunto de trabajo presenta **alta integridad estructural**.
No se elimina ni imputa ningún registro. Las únicas transformaciones que se aplican
en la Actividad 3 son el cálculo de métricas derivadas (vectorizadas) y el etiquetado de
películas sin inventario, cuya presencia en los datos está técnicamente justificada.
